In [ ]:
train_dir = '/kaggle/input/foodimages/datasets/train'
test_dir = '/kaggle/input/foodimages/datasets/test'

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [ ]:
category_to_nutri_grade = {
    'Apple': 'A',
    'Apricot': 'A',
    'banana': 'A',
    'Blackberry': 'A',
    'blueberries': 'A',
    'Papaya': 'A',
    'orange': 'A',
    'pear': 'A',
    'salad': 'A',
    'mixed vegetables': 'A',
    'green leafy vegetables': 'A',
    'sandwich': 'A',
    'salmon - grilled': 'A',
    'Soft boiled eggs': 'A',
    'milk': 'A',
    'Nuts': 'A',
    'whole grain bread': 'A',
    'whole oats': 'A',
    'cooked brown rice': 'A',
    'cooked white rice': 'A',
    'corn': 'A',
    'Porridge': 'A',
    'yogurt': 'A',
    'thunder tea rice': 'A',

    'steamed grouper': 'B',
    'Ban Mian': 'B',
    'bee hoon': 'B',
    'Udon': 'B',
    'Fish Ball Noodles': 'B',
    'Seafood Noodles Soup': 'B',
    'Prawn Noodle': 'B',
    'sirloin steak': 'B',
    'pasta - red sauce': 'B',
    'dumpling': 'B',
    'siew mai': 'B',
    'Bibimbap': 'B',
    'chicken soup': 'B',
    'muesli': 'B',
    'popiah': 'B',
    'kebab - chicken': 'B',
    'sushi': 'B',
    'roasted chicken': 'B',
    'otak': 'B',

    'Lor mee': 'C',
    'Mee rebus': 'C',
    'Mee siam': 'C',
    'nasi lemak': 'C',
    'bak kut teh': 'C',
    'Duck Rice': 'C',
    'Claypot Rice': 'C',
    'rice dumpling': 'C',
    'pineapple tarts': 'C',
    'Miso ramen, with fishcake': 'C',
    'chwee kueh': 'C',
    'chicken rice': 'C',
    'Hor Fun': 'C',
    'hokkien prawn mee': 'C',
    'goreng pisang': 'C',
    'tacos and nachos': 'C',

    'Burger': 'D',
    'sambal stingray': 'D',
    'oyster omelette': 'D',
    'cheese fries': 'D',
    'bak kwa': 'D',
    'chilli crab': 'D',
    'black pepper crab': 'D',
    'fish head curry': 'D',
    'Indian Prata': 'D',
    'ayam penyet': 'D',
    'Kway Teow': 'D',
    'Fish and chips': 'D',
    'fried chicken': 'D',
    'har cheong gai': 'D',
    'satay bee hoon': 'D',
    'ice kacang': 'D',
    'Laksa': 'D',
    'Chinese fritters': 'D',
    'curry puff': 'D'
}

nutri_grade_to_numeric = {
    'A': 0,
    'B': 1,
    'C': 2,
    'D': 3
}

Add 1 FC layer (78, 4) to 4 Nutri-Grade levels

In [ ]:
from torch.utils.data import Dataset

class NutriGradeDataset(Dataset):
    def __init__(self, dataset, category_to_nutri_grade, nutri_grade_to_numeric):
        self.dataset = dataset
        self.category_to_nutri_grade = category_to_nutri_grade
        self.nutri_grade_to_numeric = nutri_grade_to_numeric

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        class_name = self.dataset.classes[label]
        nutri_grade = self.category_to_nutri_grade[class_name]
        nutri_idx = self.nutri_grade_to_numeric[nutri_grade]

        return img, nutri_idx

In [ ]:
import os
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

class VanillaCNN(nn.Module):
    def __init__(self, num_classes):
        super(VanillaCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 112 * 112, 128)
        self.fc2 = nn.Linear(128, 78)
        self.fc3 = nn.Linear(78,num_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
from torch.utils.data import DataLoader, random_split
def load_data(train_dir, test_dir, batch_size):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        # normalize to match with pretrained value
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    data = datasets.ImageFolder(train_dir, transform=transform)
    data_converted = NutriGradeDataset(data, category_to_nutri_grade, nutri_grade_to_numeric)

    val_size = int(0.2 * len(data_converted))
    train_size = len(data_converted) - val_size
    train_data, val_data = random_split(data_converted, [train_size, val_size])

    test_data = datasets.ImageFolder(test_dir, transform=transform)
    test_data_converted = NutriGradeDataset(test_data, category_to_nutri_grade, nutri_grade_to_numeric)
    train_loader = DataLoader(train_data, batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_data, batch_size, shuffle=False, num_workers=2)

    test_loader  = DataLoader(test_data_converted, batch_size)
    return train_loader, val_loader, test_loader

In [ ]:
import time
from torch.amp import autocast, GradScaler
def train(model, train_loader, val_loader, criterion, optimizer, epochs):
    scaler = GradScaler()
    min_val_loss = float('inf')
    patience = 2
    trigger_times = 0
    model.to(device)
    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        training_loss = 0.0
        for i, l in train_loader:
            i, l = i.to(device), l.to(device)
            optimizer.zero_grad()
            # reduce precision for less mem & faster training
            with autocast(device_type='cuda'):
                output = model(i)
                loss = criterion(output, l)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            training_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for i, l in val_loader:
                i, l = i.to(device), l.to(device)
                with autocast(device_type='cuda'):
                    outputs = model(i)
                    loss = criterion(outputs, l)
                val_loss += loss.item()
        avg_val_loss = val_loss / len(val_loader)
        epoch_time = time.time() - start_time
        print(f'epoch [{epoch+1}/{epochs}], train loss: {training_loss/len(train_loader):.4f}  | val loss: {val_loss/len(val_loader):.4f}  | '
              f'time: {epoch_time:.2f}s')

        # early stopping
        if avg_val_loss < min_val_loss:
            min_val_loss = avg_val_loss
            best_model_wts = model.state_dict()
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print("early stopping")
                break

    model.load_state_dict(best_model_wts)
    return model


In [ ]:
def evaluate(model, test_loader, criterion):
    model.to(device)
    model.eval()
    correct = 0
    total = 0
    test_loss = 0.0

    with torch.no_grad():
        for i, l in test_loader:
            i, l = i.to(device), l.to(device)
            output = model(i)
            loss = criterion(output, l)
            test_loss += loss.item()

            _, predicted = torch.max(output.data, 1)
            total += l.size(0)
            correct += (predicted == l).sum().item()

    accuracy = correct * 100 / total
    avg_loss = test_loss / len(test_loader)

    print(f'test loss: {avg_loss:.4f} | test accuracy: {accuracy:.4f}%')

In [ ]:
num_classes = 4
train_loader, val_loader, test_loader = load_data(train_dir, test_dir, 64)
criterion = nn.CrossEntropyLoss()

In [ ]:
model = VanillaCNN(num_classes)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
train(model, train_loader, val_loader, criterion, optimizer, 10)
evaluate(model, test_loader, criterion)

epoch [1/10], train loss: 1.3763  | val loss: 1.2484  | time: 148.30s
epoch [2/10], train loss: 1.2035  | val loss: 1.2498  | time: 93.46s
epoch [3/10], train loss: 1.1491  | val loss: 1.2386  | time: 91.50s
epoch [4/10], train loss: 1.0975  | val loss: 1.2165  | time: 95.81s
epoch [5/10], train loss: 1.0367  | val loss: 1.2089  | time: 92.15s
epoch [6/10], train loss: 0.9647  | val loss: 1.2389  | time: 93.06s
epoch [7/10], train loss: 0.8679  | val loss: 1.2439  | time: 92.32s
early stopping
test loss: 1.2410 | test accuracy: 46.0800%


In [ ]:
torch.save(model.state_dict(), 'VanillaCnn_Nutri_Model_78_4_model.pth')

lenet5

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, num_classes):
        super(LeNet5, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5, stride=1, padding=2)
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1)
        self.conv3 = nn.Conv2d(16, 120, kernel_size=5, stride=1)
        self.fc1 = nn.Linear(120 * 50 * 50, 84)
        self.fc2 = nn.Linear(84, 78)
        self.fc3 = nn.Linear(78,num_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.relu(self.conv3(x))
        x = torch.flatten(x,1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
Lenet5Model = LeNet5(num_classes)
optimizer = optim.Adam(Lenet5Model.parameters(), lr=1e-4)
train(Lenet5Model, train_loader, val_loader, criterion, optimizer, 10)
evaluate(Lenet5Model, test_loader, criterion)

epoch [1/10], train loss: 1.2599  | val loss: 1.2311  | time: 153.14s
epoch [2/10], train loss: 1.1823  | val loss: 1.2026  | time: 97.60s
epoch [3/10], train loss: 1.1127  | val loss: 1.1842  | time: 94.96s
epoch [4/10], train loss: 1.0197  | val loss: 1.2042  | time: 96.91s
epoch [5/10], train loss: 0.8755  | val loss: 1.2619  | time: 102.41s
early stopping
test loss: 1.2352 | test accuracy: 46.1189%


In [ ]:
torch.save(model.state_dict(), 'Lenet5_Nutri_Model_78_4_model.pth')

Vgg16

In [ ]:
import torchvision.models as models
# Load the VGG16 model
class VGG16(nn.Module):
    def __init__(self, num_classes):
        super(VGG16, self).__init__()
        self.model = models.vgg16(pretrained=True)
        self.model.classifier[6] = nn.Linear(4096, 78)
        self.fc = nn.Linear(78, num_classes)

    def forward(self, x):
        x = self.model(x)
        x = self.fc(x)
        return x

In [ ]:
Vgg16Model = VGG16(num_classes)
optimizer = optim.Adam(Vgg16Model.parameters(), lr=1e-4)
train(Vgg16Model, train_loader, val_loader, criterion, optimizer, 10)
evaluate(Vgg16Model, test_loader, criterion)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:02<00:00, 207MB/s] 


epoch [1/10], train loss: 0.8795  | val loss: 0.7349  | time: 234.99s
epoch [2/10], train loss: 0.6062  | val loss: 0.6104  | time: 235.23s
epoch [3/10], train loss: 0.4214  | val loss: 0.6223  | time: 235.46s
epoch [4/10], train loss: 0.2820  | val loss: 0.6490  | time: 234.70s
early stopping
test loss: 0.6703 | test accuracy: 77.7648%


In [ ]:
torch.save(Vgg16Model.state_dict(), 'Vgg16_Nutri_Model_78_4_model.pth')

To 4 Nutri-Grade levels directly

In [ ]:
class VanillaCNN4(nn.Module):
    def __init__(self, num_classes):
        super(VanillaCNN4, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 112 * 112, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
VanillaCnnModel4 = VanillaCNN4(num_classes)
optimizer = optim.Adam(VanillaCnnModel4.parameters(), lr=1e-4)
train(VanillaCnnModel4, train_loader, val_loader, criterion, optimizer, 10)
evaluate(VanillaCnnModel4, test_loader, criterion)

epoch [1/10], train loss: 1.5250  | val loss: 1.2594  | time: 114.11s
epoch [2/10], train loss: 1.2016  | val loss: 1.2577  | time: 95.64s
epoch [3/10], train loss: 1.1257  | val loss: 1.2134  | time: 94.55s
epoch [4/10], train loss: 1.0600  | val loss: 1.2394  | time: 87.57s
epoch [5/10], train loss: 0.9880  | val loss: 1.2395  | time: 91.01s
early stopping
test loss: 1.2299 | test accuracy: 44.3406%


In [ ]:
torch.save(VanillaCnnModel4.state_dict(), 'VanillaCnn_Nutri_Model_4_model.pth')

In [ ]:
class LeNet5_4(nn.Module):
    def __init__(self, num_classes):
        super(LeNet5_4, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5, stride=1, padding=2)
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1)
        self.conv3 = nn.Conv2d(16, 120, kernel_size=5, stride=1)
        self.fc1 = nn.Linear(120 * 50 * 50, 84)
        self.fc2 = nn.Linear(84,num_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.relu(self.conv3(x))
        x = torch.flatten(x,1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
Lenet5Model4 = LeNet5_4(num_classes)
optimizer = optim.Adam(Lenet5Model4.parameters(), lr=1e-4)
train(Lenet5Model4, train_loader, val_loader, criterion, optimizer, 10)
evaluate(Lenet5Model4, test_loader, criterion)

epoch [1/10], train loss: 1.2444  | val loss: 1.2089  | time: 113.24s
epoch [2/10], train loss: 1.1499  | val loss: 1.1768  | time: 91.89s
epoch [3/10], train loss: 1.0457  | val loss: 1.2044  | time: 91.17s
epoch [4/10], train loss: 0.9098  | val loss: 1.2542  | time: 92.76s
early stopping
test loss: 1.2456 | test accuracy: 46.6900%


In [ ]:
torch.save(Lenet5Model4.state_dict(), 'Lenet5_Nutri_Model_4_model.pth')

In [ ]:
class VGG16_4(nn.Module):
    def __init__(self, num_classes):
        super(VGG16_4, self).__init__()
        self.model = models.vgg16(pretrained=True)
        self.model.classifier[6] = nn.Linear(4096, num_classes)

    def forward(self, x):
        return self.model(x)

In [ ]:
Vgg16Model4 = VGG16_4(num_classes)
optimizer = optim.Adam(Vgg16Model4.parameters(), lr=1e-4)
train(Vgg16Model4, train_loader, val_loader, criterion, optimizer, 10)
evaluate(Vgg16Model4, test_loader, criterion)

epoch [1/10], train loss: 0.9023  | val loss: 0.7450  | time: 238.75s
epoch [2/10], train loss: 0.6174  | val loss: 0.6537  | time: 236.54s
epoch [3/10], train loss: 0.4464  | val loss: 0.6955  | time: 236.25s
epoch [4/10], train loss: 0.2895  | val loss: 0.6892  | time: 235.15s
early stopping
test loss: 0.7112 | test accuracy: 75.9476%


In [ ]:
torch.save(Vgg16Model4.state_dict(), 'Vgg16_Nutri_Model_4_model.pth')